In [ ]:
#Importing necessary libraries:

import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import re
from pygam import LinearGAM, s, f, l
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import numpy as np
import scipy.stats as stats


In [ ]:
#Loading the data into dataframe

df = pd.read_stata("data/UKB 71392 CMD Project.dta")


print(df.shape)

list(df.columns)

In [ ]:
#Checking datatypes and nulls
df.info()

#Stats of numerical columns
df.describe()

#Number of missing values/column
df.isnull().sum()

list(df.columns)

In [ ]:
#Explorng PRS ADHD data 
df['prs_adhd_std'].describe()

In [ ]:
#Visualising the distribution of PRS ADHD 

plt.figure(figsize=(8, 4))
sns.histplot(df['prs_adhd_std'].dropna(), kde=True)
plt.title('Distribution of Standardized ADHD PRS')
plt.xlabel('ADHD PRS (standardized)')
plt.ylabel('Frequency')
plt.show()

stats.probplot(df['prs_adhd_std'].dropna(), dist="norm", plot=plt)
plt.title('QQ Plot of Standardized ADHD PRS')
plt.show()

In [ ]:
#Cleaning the dataset 

#Fixing the sex column and turning it binary:
df['sex'] = df['sex_plink'].map({1: 0, 2: 1}) #0 => male and 1=> female

#Removing individuals with missing PRS and Key covariates:
main_vars = ['prs_adhd_std', 'n_21001_0_0', 'age_0_0', 'sex',
    'townsend', 'cursmoke_0', 'curr_alcohol_0',
    'PGC1', 'PGC2', 'PGC3', 'PGC4', 'PGC5', 'PGC6', 'PGC7', 'PGC8',
    'chip','whr_0']

df_model = df.dropna(subset=main_vars).copy()

print(df_model.shape)
list(df_model.columns)

In [ ]:
#Visualizing the WHR distribution 

plt.figure(figsize=(10, 6))
sns.histplot(df_model['whr_0'], kde=True, bins=50, color='skyblue')
plt.title("Distribution of Waist Hip Ratio (WHR)")
plt.xlabel("WHR")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# exploring the WHR distribution:
df['whr_0'].describe()
df['whr_0'].quantile([0.001, 0.005, 0.01, 0.05, 0.5, 0.95, 0.99, 0.995, 0.999])

In [ ]:
#cleaning up WHR

lower = df['whr_0'].quantile(0.001)
upper = df['whr_0'].quantile(0.999)

df_whr_clean = df[
    (df['whr_0'] >= lower) &
    (df['whr_0'] <= upper)
].copy()

print(lower, upper)
print(df.shape)
print(df_whr_clean.shape)


In [ ]:
#plotting distribution of cleaned WHR

df_whr_clean['whr_0'].describe()

plt.figure(figsize=(8, 5))
sns.histplot(df_whr_clean['whr_0'], bins=50, kde=True)
plt.xlabel('Waist-to-hip ratio')
plt.ylabel('Frequency')
plt.title('Distribution of Waist-to-Hip Ratio after Outlier Removal')
plt.show()

In [ ]:
#Scaling Age and SES factors

scale_cols = ['age_0_0', 'townsend']
scaler = StandardScaler()
df_whr_clean[scale_cols] = scaler.fit_transform(df_whr_clean[scale_cols])

In [ ]:
#Plotting the WHR distribution into PRS tertiles to visualise trends:

df_whr_clean['prs_tertile'] = pd.qcut(df_whr_clean['prs_adhd_std'], q =3, labels = ["Low","medium","high"])

# Boxplot of WHR by PRS tertile

plt.figure(figsize=(8, 5))
sns.boxplot(x='prs_tertile', y='whr_0', data=df_whr_clean,  palette='Set2')
plt.title('WHR across ADHD PRS Tertiles')
plt.xlabel('ADHD PRS Tertile')
plt.ylabel('WHR')
plt.show()


# Print mean and median WHR for each PRS tertile
WHR_tertile_summary = (
    df_whr_clean
    .groupby('prs_tertile')['whr_0']
    .agg(['count', 'mean', 'std', 'median', 'min', 'max'])
    
)

print(WHR_tertile_summary)


In [ ]:
#Narrow Y-axis around the WHR range
df_whr_clean['prs_tertile'] = pd.qcut(
    df_whr_clean['prs_adhd_std'],
    q=3,
    labels=['Low', 'Medium', 'High']
)

plt.figure(figsize=(7, 5))

sns.boxplot(
    x='prs_tertile',
    y='whr_0',
    data=df_whr_clean,
    showfliers=False
)

plt.ylim(0.78, 0.96)
plt.title('WHR across ADHD PRS Tertiles')
plt.xlabel('ADHD PRS Tertile')
plt.ylabel('Waist-to-hip ratio')
plt.show()

In [ ]:
#Add mean points on top of boxplot

plt.figure(figsize=(7, 5))

sns.boxplot(
    x='prs_tertile',
    y='whr_0',
    data=df_whr_clean,
    showfliers=False
)

sns.pointplot(
    x='prs_tertile',
    y='whr_0',
    data=df_whr_clean,
    estimator='mean',
    errorbar=None,
    markers='D',
    linestyles='-'
)

plt.ylim(0.78, 0.96)
plt.title('WHR across ADHD PRS Tertiles')
plt.xlabel('ADHD PRS Tertile')
plt.ylabel('Waist-to-hip ratio')
plt.show()

In [ ]:
#plotting median WHR only

median_whr = (
    df_whr_clean
    .groupby('prs_tertile', observed=True)['whr_0']
    .median()
    .reset_index()
)

plt.figure(figsize=(6, 4))

sns.pointplot(
    x='prs_tertile',
    y='whr_0',
    data=median_whr,
    markers='o',
    linestyles='-'
)

plt.title('Median WHR across ADHD PRS Tertiles')
plt.xlabel('ADHD PRS Tertile')
plt.ylabel('Median waist-to-hip ratio')
plt.show()

median_whr

In [ ]:
# Check whether ADHD PRS differs by sex

df_sex_check = df_whr_clean[
    ['prs_adhd_std', 'sex', 'whr_0']
].replace([np.inf, -np.inf], np.nan).dropna().copy()

df_sex_check['sex'] = df_sex_check['sex'].astype('category')

# Summary statistics by sex
sex_prs_summary = df_sex_check.groupby('sex')['prs_adhd_std'].agg(
    n='count',
    mean='mean',
    median='median',
    std='std'
)

sex_whr_summary = df_sex_check.groupby('sex')['whr_0'].agg(
    n='count',
    mean='mean',
    median='median',
    std='std'
)

print("ADHD PRS by sex")
print(sex_prs_summary)

print("\nWHR by sex")
print(sex_whr_summary)

In [ ]:
# Residualise WHR for sex, then test ADHD PRS association

import statsmodels.formula.api as smf

df_resid = df_whr_clean[
    [
        'prs_adhd_std', 'whr_0', 'sex',
        'age_0_0', 'townsend', 'cursmoke_0', 'curr_alcohol_0', 'chip',
        'PGC1', 'PGC2', 'PGC3', 'PGC4',
        'PGC5', 'PGC6', 'PGC7', 'PGC8'
    ]
].replace([np.inf, -np.inf], np.nan).dropna().copy()

df_resid['sex'] = df_resid['sex'].astype('category')
df_resid['chip'] = df_resid['chip'].astype('category')
df_resid['cursmoke_0'] = df_resid['cursmoke_0'].astype('category')
df_resid['curr_alcohol_0'] = df_resid['curr_alcohol_0'].astype('category')

# Step 1: remove sex effect from WHR
sex_model = smf.ols(
    formula='whr_0 ~ sex',
    data=df_resid
).fit()

df_resid['whr_sex_residual'] = sex_model.resid

# Step 2: test whether ADHD PRS predicts sex-residualised WHR
resid_model = smf.ols(
    formula="""
    whr_sex_residual ~ prs_adhd_std
                     + age_0_0
                     + townsend
                     + C(cursmoke_0)
                     + C(curr_alcohol_0)
                     + PGC1 + PGC2 + PGC3 + PGC4
                     + PGC5 + PGC6 + PGC7 + PGC8
                     + C(chip)
    """,
    data=df_resid
).fit()

print(resid_model.summary())

In [ ]:
# Model 1: BMI~ ADHD PRS + genetic

#Filter and select relevant columns
cols = ['prs_adhd_std', 'whr_0', 'chip',
        'PGC1', 'PGC2', 'PGC3', 'PGC4',
        'PGC5', 'PGC6', 'PGC7', 'PGC8']
df_model1 = df_whr_clean[cols].dropna().copy()

#Define predictors (X) and outcome (y)
X = df_model1.drop(columns='whr_0')
X = pd.get_dummies(X, columns=['chip'], drop_first=True, dtype=float)
X = X.apply(pd.to_numeric, errors='coerce')
X = X.dropna()
y = df_model1.loc[X.index, 'whr_0']

#Add intercept and fit OLS model
X = sm.add_constant(X)
model1 = sm.OLS(y, X).fit()
print(model1.summary())

#Predict and attach predictions back to dataframe
df_model1 = df_model1.loc[X.index]
df_model1['whr_predicted'] = model1.predict(X)

#Bin the PRS for smoother plotting
df_model1['prs_bin'] = pd.cut(df_model1['prs_adhd_std'], bins=100)
binned = df_model1.groupby('prs_bin')[['prs_adhd_std', 'whr_predicted']].mean().dropna()

#Plot
plt.figure(figsize=(8, 6))
sns.scatterplot(x='prs_adhd_std', y='whr_0', data=df_model1,
                alpha=0.1, edgecolor=None, label='Raw Data')

#Smoothed OLS fit
sns.lineplot(x='prs_adhd_std', y='whr_predicted', data=binned,
        color='red', label='OLS Fit (Smoothed)')

plt.title('Model 1: ADHD PRS + Genetic Covariates vs WHR')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('WHR')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
#Model 2: Model 1+ Age + Sex

#Filter and select relevant columns
cols = ['prs_adhd_std', 'whr_0', 'chip', 'age_0_0', 'sex_plink',
        'PGC1', 'PGC2', 'PGC3', 'PGC4', 'PGC5', 'PGC6', 'PGC7', 'PGC8']
df_model2 = df_whr_clean[cols].dropna().copy()

#Define predictors (X) and outcome (y)
X = df_model2.drop(columns='whr_0')
X = pd.get_dummies(X, columns=['chip'], drop_first=True, dtype=float)
X = X.apply(pd.to_numeric, errors='coerce')
X = X.dropna()
y = df_model2.loc[X.index, 'whr_0']

#Add intercept and fit OLS model
X = sm.add_constant(X)
model2 = sm.OLS(y, X).fit()
print(model2.summary())

#Predict and attach predictions back to dataframe
df_model2 = df_model2.loc[X.index]
df_model2['whr_predicted'] = model2.predict(X)

#Bin the PRS for smoother plotting
df_model2['prs_bin'] = pd.cut(df_model2['prs_adhd_std'], bins=100)
binned = df_model2.groupby('prs_bin')[['prs_adhd_std', 'whr_predicted']].mean().dropna()

#Plot
plt.figure(figsize=(8, 6))
sns.scatterplot(x='prs_adhd_std', y='whr_0', data=df_model2,
                alpha=0.1, edgecolor=None, label='Raw Data')

# Smoothed OLS fit
sns.lineplot(x='prs_adhd_std', y='whr_predicted', data=binned,
                color='red', label='OLS Fit (Smoothed)')
plt.title('Model 2: ADHD PRS + Genetic Covariates + Age + Sex vs WHR')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('WHR')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
#Model 3: Model 2 + lifestyle

#Select relevant columns
cols = ['prs_adhd_std', 'whr_0', 'age_0_0', 'sex', 'townsend', 
        'cursmoke_0', 'curr_alcohol_0',
        'PGC1', 'PGC2', 'PGC3', 'PGC4', 'PGC5', 'PGC6', 'PGC7', 'PGC8']
df_model3 = df_whr_clean[cols].dropna().copy()

#Define predictors and outcome
X3 = df_model3.drop(columns='whr_0')
X3 = pd.get_dummies(X3, drop_first=True, dtype=float)
X3 = X3.apply(pd.to_numeric, errors='coerce')
X3 = X3.dropna()
y = df_model3.loc[X3.index, 'whr_0']

#Add intercept and fit the model
X3 = sm.add_constant(X3)
model3 = sm.OLS(y, X3).fit()

#Print summary statistics
print(model3.summary())

#Predict and attach to dataframe
df_model3 = df_model3.loc[X3.index]
df_model3['whr_predicted'] = model3.predict(X3)

#Bin the PRS values for smoothing
df_model3['prs_bin'] = pd.cut(df_model3['prs_adhd_std'], bins=100)
mean_vals = df_model3.groupby('prs_bin')[['prs_adhd_std', 'whr_predicted']].mean().dropna()

#Plot
plt.figure(figsize=(10, 7))

#Scatterplot of actual BMI values
sns.scatterplot(x='prs_adhd_std', y='whr_0', data=df_model3, alpha=0.1, edgecolor=None, label='Observed WHR')

#Lineplot of smoothed predicted BMI
sns.lineplot(x=mean_vals['prs_adhd_std'], y=mean_vals['whr_predicted'], color='red', label='Model 3 Fit (Smoothed)')

#Labels and formatting
plt.title("Model 3: ADHD PRS + Genetic + Demographic + Lifestyle vs WHR")
plt.xlabel("Standardized ADHD PRS")
plt.ylabel("WHR")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Model 4: Model 3 + BMI

# Select relevant columns
cols = ['prs_adhd_std', 'whr_0', 'n_21001_0_0', 'age_0_0', 'sex', 'townsend', 
        'cursmoke_0', 'curr_alcohol_0',
        'PGC1', 'PGC2', 'PGC3', 'PGC4', 'PGC5', 'PGC6', 'PGC7', 'PGC8']

df_model4 = df_whr_clean[cols].dropna().copy()

# Optional: rename BMI column for cleaner output
df_model4 = df_model4.rename(columns={'n_21001_0_0': 'BMI'})

# Define predictors and outcome
X4 = df_model4.drop(columns='whr_0')
X4 = pd.get_dummies(X4, drop_first=True, dtype=float)
X4 = X4.apply(pd.to_numeric, errors='coerce')
X4 = X4.dropna()

y = df_model4.loc[X4.index, 'whr_0']

# Add intercept and fit the model
X4 = sm.add_constant(X4)
model4 = sm.OLS(y, X4).fit()

# Print summary statistics
print(model4.summary())

# Predict and attach to dataframe
df_model4 = df_model4.loc[X4.index].copy()
df_model4['whr_predicted'] = model4.predict(X4)

# Bin the PRS values for smoothing
df_model4['prs_bin'] = pd.cut(df_model4['prs_adhd_std'], bins=100)

mean_vals = (
    df_model4
    .groupby('prs_bin', observed=True)[['prs_adhd_std', 'whr_predicted']]
    .mean()
    .dropna()
)

# Plot
plt.figure(figsize=(10, 7))

# Scatterplot of observed WHR values
sns.scatterplot(
    x='prs_adhd_std',
    y='whr_0',
    data=df_model4,
    alpha=0.1,
    edgecolor=None,
    label='Observed WHR'
)

# Lineplot of smoothed predicted WHR
sns.lineplot(
    x=mean_vals['prs_adhd_std'],
    y=mean_vals['whr_predicted'],
    color='red',
    label='Model 4 Fit (Smoothed)'
)

# Labels and formatting
plt.title("Model 4: ADHD PRS + Genetic + Demographic + Lifestyle + BMI vs WHR")
plt.xlabel("Standardized ADHD PRS")
plt.ylabel("WHR")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#does ADHD PRS effect on WHR change between sexes?
# Testing whether the association between ADHD PRS and WHR differs by sex
# Key term: prs_adhd_std*sex

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# Select relevant columns
cols = [
    'prs_adhd_std',
    'whr_0',
    'age_0_0',
    'sex',
    'townsend',
    'cursmoke_0',
    'curr_alcohol_0',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

df_interaction = df_whr_clean[cols].copy()

# Remove missing and infinite values
df_interaction = df_interaction.replace([np.inf, -np.inf], np.nan).dropna().copy()

# Treat categorical variables properly
df_interaction['sex'] = df_interaction['sex'].astype('category')
df_interaction['chip'] = df_interaction['chip'].astype('category')
df_interaction['cursmoke_0'] = df_interaction['cursmoke_0'].astype('category')
df_interaction['curr_alcohol_0'] = df_interaction['curr_alcohol_0'].astype('category')

# Fit OLS model with ADHD PRS x sex interaction
interaction_model = smf.ols(
    formula="""
    whr_0 ~ prs_adhd_std * sex
          + age_0_0
          + townsend
          + C(cursmoke_0)
          + C(curr_alcohol_0)
          + PGC1 + PGC2 + PGC3 + PGC4
          + PGC5 + PGC6 + PGC7 + PGC8
          + C(chip)
    """,
    data=df_interaction
).fit()

# Print full model summary
print(interaction_model.summary())


END OF OLS ANALSYSIS

In [ ]:
# GAM 1: WHR ~ ADHD PRS + genetic covariates + chip

from pygam import LinearGAM, s, f
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Select columns
cols = [
    'prs_adhd_std',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8',
    'chip',
    'whr_0'
]

df_gam1 = df_whr_clean[cols].copy()

# Replace inf values with NaN
df_gam1 = df_gam1.replace([np.inf, -np.inf], np.nan)

# Drop missing values from both predictors and outcome
df_gam1 = df_gam1.dropna().copy()

# Convert chip to categorical numeric codes if needed
df_gam1['chip'] = df_gam1['chip'].astype('category').cat.codes

# Prepare X and y
X_gam1 = df_gam1[
    [
        'prs_adhd_std',
        'PGC1', 'PGC2', 'PGC3', 'PGC4',
        'PGC5', 'PGC6', 'PGC7', 'PGC8',
        'chip'
    ]
].values

y_gam1 = df_gam1['whr_0'].values

# Check that there are no NaN or inf values left
print("NaN in X:", np.isnan(X_gam1).sum())
print("Inf in X:", np.isinf(X_gam1).sum())
print("NaN in y:", np.isnan(y_gam1).sum())
print("Inf in y:", np.isinf(y_gam1).sum())
print("Shape:", X_gam1.shape, y_gam1.shape)

# Fit GAM
gam1 = LinearGAM(
    s(0) + s(1) + s(2) + s(3) + s(4) +
    s(5) + s(6) + s(7) + s(8) +
    f(9)
).fit(X_gam1, y_gam1)

# Generate PRS grid for term 0
XX = gam1.generate_X_grid(term=0)

# Predict adjusted WHR and confidence intervals
pdep = gam1.predict(XX)
conf_int = gam1.confidence_intervals(XX)

# Plot
plt.figure(figsize=(10, 6))

plt.plot(
    XX[:, 0],
    pdep,
    color='red',
    label='GAM fit'
)

plt.fill_between(
    XX[:, 0],
    conf_int[:, 0],
    conf_int[:, 1],
    alpha=0.3,
    color='blue',
    label='95% CI'
)

plt.title("GAM 1: ADHD PRS and WHR adjusted for genetic covariates")
plt.xlabel("Standardised ADHD PRS")
plt.ylabel("Predicted WHR")
plt.legend()
plt.tight_layout()
plt.show()

# Print summary
gam1.summary()

In [ ]:
# GAM 2: WHR ~ ADHD PRS + genetic covariates + chip + age + sex

cols = [
    'prs_adhd_std',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8',
    'chip',
    'age_0_0',
    'sex',
    'whr_0'
]

df_gam2 = df_whr_clean[cols].copy()
df_gam2 = df_gam2.replace([np.inf, -np.inf], np.nan).dropna().copy()

# Convert categorical variables
df_gam2['chip'] = df_gam2['chip'].astype('category').cat.codes
df_gam2['sex'] = df_gam2['sex'].astype('category').cat.codes

# Prepare X and y
X_gam2 = df_gam2[
    [
        'prs_adhd_std',
        'PGC1', 'PGC2', 'PGC3', 'PGC4',
        'PGC5', 'PGC6', 'PGC7', 'PGC8',
        'chip',
        'age_0_0',
        'sex'
    ]
].values

y_gam2 = df_gam2['whr_0'].values

# Fit GAM
gam2 = LinearGAM(
    s(0) +                                  # ADHD PRS
    s(1) + s(2) + s(3) + s(4) +
    s(5) + s(6) + s(7) + s(8) +             # PCs
    f(9) +                                  # chip
    s(10) +                                 # age
    f(11)                                   # sex
).fit(X_gam2, y_gam2)

# Generate PRS grid
XX = gam2.generate_X_grid(term=0)

# Predict adjusted WHR and CI
pdep = gam2.predict(XX)
conf_int = gam2.confidence_intervals(XX)

# Plot
plt.figure(figsize=(10, 6))

plt.plot(
    XX[:, 0],
    pdep,
    color='red',
    label='GAM fit'
)

plt.fill_between(
    XX[:, 0],
    conf_int[:, 0],
    conf_int[:, 1],
    alpha=0.3,
    color='blue',
    label='95% CI'
)

plt.title("GAM 2: ADHD PRS and WHR adjusted for genetic covariates, age and sex")
plt.xlabel("Standardised ADHD PRS")
plt.ylabel("Predicted WHR")
plt.legend()
plt.tight_layout()
plt.show()

gam2.summary()

In [ ]:
# GAM 3: WHR ~ ADHD PRS + genetic + demographic + lifestyle covariates

cols = [
    'prs_adhd_std',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8',
    'chip',
    'age_0_0',
    'sex',
    'townsend',
    'cursmoke_0',
    'curr_alcohol_0',
    'whr_0'
]

df_gam3 = df_whr_clean[cols].copy()
df_gam3 = df_gam3.replace([np.inf, -np.inf], np.nan).dropna().copy()

# Convert categorical variables
df_gam3['chip'] = df_gam3['chip'].astype('category').cat.codes
df_gam3['sex'] = df_gam3['sex'].astype('category').cat.codes
df_gam3['cursmoke_0'] = df_gam3['cursmoke_0'].astype('category').cat.codes
df_gam3['curr_alcohol_0'] = df_gam3['curr_alcohol_0'].astype('category').cat.codes

# Prepare X and y
X_gam3 = df_gam3[
    [
        'prs_adhd_std',
        'PGC1', 'PGC2', 'PGC3', 'PGC4',
        'PGC5', 'PGC6', 'PGC7', 'PGC8',
        'chip',
        'age_0_0',
        'sex',
        'townsend',
        'cursmoke_0',
        'curr_alcohol_0'
    ]
].values

y_gam3 = df_gam3['whr_0'].values

# Fit GAM
gam3 = LinearGAM(
    s(0) +                                  # ADHD PRS
    s(1) + s(2) + s(3) + s(4) +
    s(5) + s(6) + s(7) + s(8) +             # PGC1 to PGC8
    f(9) +                                  # chip
    s(10) +                                 # age
    f(11) +                                 # sex
    s(12) +                                 # townsend
    f(13) +                                 # current smoking
    f(14)                                   # current alcohol
).fit(X_gam3, y_gam3)

# Generate PRS grid for ADHD PRS term
XX = gam3.generate_X_grid(term=0)

# Predict adjusted WHR and confidence intervals
pdep = gam3.predict(XX)
conf_int = gam3.confidence_intervals(XX)

# Plot
plt.figure(figsize=(10, 6))

plt.plot(
    XX[:, 0],
    pdep,
    color='red',
    label='GAM fit'
)

plt.fill_between(
    XX[:, 0],
    conf_int[:, 0],
    conf_int[:, 1],
    alpha=0.3,
    color='blue',
    label='95% CI'
)

plt.title("GAM 3: ADHD PRS and WHR adjusted for genetic, demographic and lifestyle covariates")
plt.xlabel("Standardised ADHD PRS")
plt.ylabel("Predicted WHR")
plt.legend()
plt.tight_layout()
plt.show()

gam3.summary()

In [ ]:
# GAM 4: WHR ~ ADHD PRS + genetic + demographic + lifestyle covariates + BMI

cols = [
    'prs_adhd_std',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8',
    'chip',
    'age_0_0',
    'sex',
    'townsend',
    'cursmoke_0',
    'curr_alcohol_0',
    'n_21001_0_0',   # BMI
    'whr_0'
]

df_gam4 = df_whr_clean[cols].copy()
df_gam4 = df_gam4.replace([np.inf, -np.inf], np.nan).dropna().copy()

# Rename BMI column for readability
df_gam4 = df_gam4.rename(columns={'n_21001_0_0': 'BMI'})

# Convert categorical variables
df_gam4['chip'] = df_gam4['chip'].astype('category').cat.codes
df_gam4['sex'] = df_gam4['sex'].astype('category').cat.codes
df_gam4['cursmoke_0'] = df_gam4['cursmoke_0'].astype('category').cat.codes
df_gam4['curr_alcohol_0'] = df_gam4['curr_alcohol_0'].astype('category').cat.codes

# Prepare X and y
X_gam4 = df_gam4[
    [
        'prs_adhd_std',
        'PGC1', 'PGC2', 'PGC3', 'PGC4',
        'PGC5', 'PGC6', 'PGC7', 'PGC8',
        'chip',
        'age_0_0',
        'sex',
        'townsend',
        'cursmoke_0',
        'curr_alcohol_0',
        'BMI'
    ]
].values

y_gam4 = df_gam4['whr_0'].values

# Fit GAM
gam4 = LinearGAM(
    s(0) +                                  # ADHD PRS
    s(1) + s(2) + s(3) + s(4) +
    s(5) + s(6) + s(7) + s(8) +             # PGC1 to PGC8
    f(9) +                                  # chip
    s(10) +                                 # age
    f(11) +                                 # sex
    s(12) +                                 # townsend
    f(13) +                                 # current smoking
    f(14) +                                 # current alcohol
    s(15)                                   # BMI
).fit(X_gam4, y_gam4)

# Better GAM 4 partial dependence plot for ADHD PRS term

XX = gam4.generate_X_grid(term=0)

pdep, confi = gam4.partial_dependence(
    term=0,
    X=XX,
    width=0.95
)

plt.figure(figsize=(10, 6))

plt.plot(
    XX[:, 0],
    pdep,
    color='red',
    label='ADHD PRS smooth effect'
)

plt.fill_between(
    XX[:, 0],
    confi[:, 0],
    confi[:, 1],
    alpha=0.3,
    color='blue',
    label='95% CI'
)

plt.axhline(0, color='black', linestyle='--', linewidth=1)

plt.title("GAM 4: Partial effect of ADHD PRS on WHR after BMI adjustment")
plt.xlabel("Standardised ADHD PRS")
plt.ylabel("Partial effect on WHR")
plt.legend()
plt.tight_layout()
plt.show()

gam4.summary()

In [ ]:
# GAM sensitivity model:
# Does ADHD PRS remain associated with WHR after residualising WHR for sex?

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pygam import LinearGAM, s, f
import matplotlib.pyplot as plt

# Select relevant columns
cols = [
    'prs_adhd_std',
    'whr_0',
    'sex',
    'age_0_0',
    'townsend',
    'cursmoke_0',
    'curr_alcohol_0',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

df_gam_resid = df_whr_clean[cols].copy()
df_gam_resid = df_gam_resid.replace([np.inf, -np.inf], np.nan).dropna().copy()

# Treat sex as categorical for residualisation
df_gam_resid['sex'] = df_gam_resid['sex'].astype('category')

# Step 1: residualise WHR for sex
sex_model = smf.ols(
    formula='whr_0 ~ sex',
    data=df_gam_resid
).fit()

df_gam_resid['whr_sex_residual'] = sex_model.resid

# Convert categorical predictors for GAM
df_gam_resid['chip'] = df_gam_resid['chip'].astype('category').cat.codes
df_gam_resid['cursmoke_0'] = df_gam_resid['cursmoke_0'].astype('category').cat.codes
df_gam_resid['curr_alcohol_0'] = df_gam_resid['curr_alcohol_0'].astype('category').cat.codes

# Prepare X and y
X_gam_resid = df_gam_resid[
    [
        'prs_adhd_std',
        'PGC1', 'PGC2', 'PGC3', 'PGC4',
        'PGC5', 'PGC6', 'PGC7', 'PGC8',
        'chip',
        'age_0_0',
        'townsend',
        'cursmoke_0',
        'curr_alcohol_0'
    ]
].values

y_gam_resid = df_gam_resid['whr_sex_residual'].values

# Fit GAM
gam_resid = LinearGAM(
    s(0) +                                  # ADHD PRS
    s(1) + s(2) + s(3) + s(4) +
    s(5) + s(6) + s(7) + s(8) +             # PGC1 to PGC8
    f(9) +                                  # chip
    s(10) +                                 # age
    s(11) +                                 # townsend
    f(12) +                                 # current smoking
    f(13)                                   # current alcohol
).fit(X_gam_resid, y_gam_resid)

# Partial dependence plot for ADHD PRS
XX = gam_resid.generate_X_grid(term=0)

pdep, confi = gam_resid.partial_dependence(
    term=0,
    X=XX,
    width=0.95
)

plt.figure(figsize=(10, 6))

plt.plot(
    XX[:, 0],
    pdep,
    color='red',
    label='ADHD PRS smooth effect'
)

plt.fill_between(
    XX[:, 0],
    confi[:, 0],
    confi[:, 1],
    alpha=0.3,
    color='blue',
    label='95% CI'
)

plt.axhline(0, color='black', linestyle='--', linewidth=1)

plt.title("GAM sensitivity model: ADHD PRS effect on sex-residualised WHR")
plt.xlabel("Standardised ADHD PRS")
plt.ylabel("Partial effect on sex-residualised WHR")
plt.legend()
plt.tight_layout()
plt.show()

gam_resid.summary()